# Agent Foundations and Typed Tools

Agents are LLM systems that can choose actions, call tools, observe results, and continue toward a task goal. In this notebook, you inspect the typed tool layer of a Northstar operations assistant and run it through PydanticAI, using either a live LLM or scripted replies.

## Learning Objectives

- Distinguish model proposals from application-controlled tool execution.
- Inspect typed tool inputs and policy results for a support ticket.
- Run a PydanticAI workflow and inspect its validated decision.

## Mental Model

Follow ticket `T-1001` from the request through typed tools to the validated `AgentDecision`. Each tool result becomes input to the next model response.


```mermaid
sequenceDiagram
    participant U as User
    participant A as PydanticAI Agent
    participant M as Model
    participant T as Typed tools
    U->>A: "Plan the next action for T-1001."
    A->>M: request + tool schemas
    M-->>A: call lookup_ticket(ticket_id="T-1001")
    A->>T: lookup_ticket
    T-->>A: topic=shipping, region=EU, customer=Mira Stone
    A->>M: tool result
    M-->>A: call lookup_policy, find_available_owner, recall_memory
    A->>T: run the three tools
    T-->>A: shipping-expedite, Priya, memory notes
    A->>M: tool results
    M-->>A: final AgentDecision
    A->>A: validate AgentDecision schema
    A-->>U: owner=Priya, policy_ids=[shipping-expedite]
```

Classic AI textbooks describe the same loop: observe the environment, choose an action, and act. A **simple reflex agent** maps what it senses directly to an action through condition-action rules. A **model-based reflex agent** also keeps internal state, so earlier observations can shape the next action.

<div align="center">
  <img src="assets/01-simple-reflex-agent.png" alt="Simple reflex agent loop" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Simple reflex agent: sensors feed condition-action rules, which drive the actuators. Source: <a href="https://commons.wikimedia.org/wiki/File:Simple_reflex_agent.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

<div align="center">
  <img src="assets/01-model-based-reflex-agent.png" alt="Model-based reflex agent with internal state" width="600" style="max-width: 100%; height: auto;" />
  <p><em>Model-based reflex agent: internal state and a model of the world sit between perception and action. Source: <a href="https://commons.wikimedia.org/wiki/File:Model_based_reflex_agent.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

In an LLM agent, the model takes the place of the condition-action rules: it chooses the next action, while tools act as its sensors (looking up a ticket) and its actuators (handing work to an owner). Notebook 02 builds that state for the operations assistant as message history and durable memory.


## Load The Shared Helpers

All notebooks use the same local data and agent setup from the [`support`](support/ops_agent.py) package, so the lesson cells can focus on agent behavior instead of repeated setup.


In [ ]:
from inspect import signature

from pydantic import ValidationError
from pydantic_ai.toolsets import FunctionToolset

from support import (
    AgentDecision,
    agent_limits,
    build_agent,
    load_operations_data,
    trace_from_result,
)

## Inspect The Data And Schema

The lesson data is deliberately tiny. This lets you inspect every tool input and output instead of trusting a hidden backend.

In [ ]:
data = load_operations_data()
print(f"tickets: {len(data.tickets)}")
print(f"policies: {len(data.policies)}")
print(data.lookup_ticket("T-1001"))

A typed result schema makes the agent contract explicit. The same idea is used by frameworks when they validate model outputs before the rest of an application trusts them.

In [ ]:
# The shared output schema is used by both the framework and the evaluation.
AgentDecision(
    ticket_id="T-1001",
    next_action="Review the delayed expedited shipment.",
    owner="Priya",
    customer_update="We are reviewing the delayed expedited shipment.",
    policy_ids=["shipping-expedite"],
)

Validation is what makes the schema useful. A decision without an `owner` is rejected before any other code can act on it, which is what PydanticAI does when a model returns an incomplete answer.

In [ ]:
try:
    AgentDecision.model_validate(
        {
            "ticket_id": "T-1001",
            "next_action": "Review the delayed expedited shipment.",
            "customer_update": "We are reviewing the delayed expedited shipment.",
        }
    )
except ValidationError as error:
    print(error)

## Build The Agent And Inspect Its Tools

PydanticAI runs the actual tool loop and validates `AgentDecision`. The shared [factory](support/ops_agent.py) registers typed backend methods with `tool_plain`, which is for tools that only need their own arguments. Tools that also need run information, such as dependencies or usage, are registered with `tool` and receive a `RunContext` as their first argument.

`LLM_PROVIDER=dry-run` uses a scripted `FunctionModel` without an API key. Select a live provider in `.env` to let an LLM choose the tools and answer instead; restart the kernel after configuration changes. Both modes use the same tools, history, usage limits, and output schema. Live tool order and decisions may differ from the illustrated trace.

In [ ]:
agent = build_agent(data)
limits = agent_limits()
print("Model:", type(agent.model).__name__)
print("Output type:", AgentDecision.__name__)
toolset = agent.toolsets[0]
assert isinstance(toolset, FunctionToolset)
print("Tools:", list(toolset.tools))

Inspect the backend methods behind the registered tools. PydanticAI derives argument schemas from these annotations; the shared factory avoids repeating registration in every notebook. One tool differs from its backend method: the agent's `recall_memory` wraps `data.recall_memory` and only passes on notes with a confidence of at least `0.85` and safe content. Notebook 02 explains that filter.

In [ ]:
registered_tools = [
    data.lookup_ticket,
    data.lookup_policy,
    data.find_available_owner,
    data.recall_memory,
    data.human_review,
]
[(tool.__name__, str(signature(tool))) for tool in registered_tools]

These annotations define the contract the model receives. PydanticAI turns each signature into a JSON schema and sends it with the tool's name and description. The model proposes calls, but arguments can be invalid: PydanticAI validates them before executing the function and can ask the model to correct a rejected call.

In [ ]:
toolset.tools["find_available_owner"].tool_def.parameters_json_schema

## Run The Agent

Run the framework with `await`, which works in a notebook's existing event loop. The trace below comes from completed tool returns, not from a separately invented plan.

In [ ]:
request = "Plan the next action for T-1001."
result = await agent.run(request, usage_limits=limits)
trace = trace_from_result(result, request, case_id="demo")
trace.decision.model_dump()

In [ ]:
[(step.tool, step.reason) for step in trace.steps]

## Exercise: Build And Validate A Decision

Build the decision for the critical ticket `T-1003` yourself, using the backend tools directly instead of the agent:

- load the ticket, then retrieve its policy and an available owner (the ticket's `topic` works as the role hint),
- build an `AgentDecision` with that owner, your own `next_action`, the policy's ID in `policy_ids`, a `customer_update` that quotes the policy text, and `requires_human_review` set to `True` when the ticket's priority is `critical`,
- run the agent on the same ticket and compare `owner`, `policy_ids`, and `requires_human_review` between your decision and the agent's.

Store your decision in `manual_decision`.

In [ ]:
# @TODO: Build manual_decision for T-1003, then compare it with the agent's decision.


<details>
<summary>Show Reference Solution</summary>

```python
critical_ticket = data.lookup_ticket("T-1003")
critical_policy = data.lookup_policy(critical_ticket["topic"])
critical_owner = data.find_available_owner(
    critical_ticket["topic"], critical_ticket["region"]
)

manual_decision = AgentDecision(
    ticket_id=critical_ticket["ticket_id"],
    next_action=f"Escalate the incident to {critical_owner['owner']}.",
    owner=critical_owner["owner"],
    customer_update=critical_policy["text"],
    policy_ids=[critical_policy["policy_id"]],
    requires_human_review=critical_ticket["priority"] == "critical",
)

agent_result = await agent.run("Plan the next action for T-1003.", usage_limits=limits)
for field in ["owner", "policy_ids", "requires_human_review"]:
    print(
        f"{field}: manual={getattr(manual_decision, field)}, "
        f"agent={getattr(agent_result.output, field)}"
    )
```

</details>


Both decisions agree on all three fields: Sam (the incident commander in APAC) owns the incident, the decision cites `incident-critical`, and it needs human review. The differences are in how they got there. You set `requires_human_review` yourself by comparing the ticket's priority with `"critical"`. The agent's flag instead comes from the result of its `human_review` tool call. The agent's `customer_update` also includes a note from durable memory, because its `recall_memory` tool ran too (see notebook 02).

**Takeaway:** the model can propose actions, but the application decides which tools exist, what their schemas mean, how many calls are allowed, and which outputs require validation. A successful tool call is not proof of a good decision either: the agent still has to call the tools that provide the evidence, such as the policy, which is why notebook 04 evaluates the whole trajectory.

## Check Your Understanding

1. What does a model-based reflex agent add to a simple reflex agent?
2. In a tool-calling agent, what does the model actually produce, and which part of the system decides which tools exist and runs them?
3. Why validate an agent's final output against a schema instead of passing its text on directly?
4. An agent calls a tool successfully, but its answer is still wrong. How can that happen?

<details>
<summary>Show Reference Answers</summary>

1. Internal state, so earlier observations can shape the next action instead of only the current input.
2. The model proposes structured tool calls or a final output. The application defines the available tools and their schemas, validates the proposed arguments, executes valid calls, and sends results or validation feedback back to the model.
3. A model can omit fields or return malformed data. Validation rejects such an output before downstream code acts on it, as the decision without an `owner` showed.
4. A successful call only means the tool ran, not that the right evidence was gathered. If the agent picks the wrong tool, or skips a source-of-truth tool such as a policy lookup, it can still produce a confident answer. That is why the whole trajectory matters, not only whether each call succeeded.

</details>


## Further Reading

- [PydanticAI Agents](https://pydantic.dev/docs/ai/core-concepts/agent/)
- [PydanticAI Function Tools](https://pydantic.dev/docs/ai/tools-toolsets/tools/)
